In [81]:
import numpy as np
import matplotlib.pyplot as plt
import random
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
random.seed(20)
np.random.seed(10)

### AdaGrad

 https://optimization-cbe-cornell-edu.translate.goog/index.php?title=AdaGrad&_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc

In [82]:
def minibatch_gradient_adagrad(X,y,epochs,N,mb,alpha_0,epsilon=1e-8):
    """
    X: training examples
    y: training labels (y_noisy/funcao observavel)
    epochs: number of epochs
    N: number of examples
    mb: minibatch size
    alpha_0: initial learning rate/step size
    epsilon: escalar que evita divisão por 0 na atualização
    """
    i_max = epochs * (N // mb)
    min_error = 1e-3
    error = 1

    m = X.shape[1] #number of attributes
    a = 10 * np.ones(m).reshape(m,1) # initial column vector of weights

    a_hist = np.zeros((i_max,m)) # weight history
    mse_hist = np.zeros((i_max)) # error history
    update_hist = np.zeros((i_max,m))      # learning rate * gradient
    v_hist = np.zeros((i_max,m))
    it = 0
    it_epoch = 0
    epoch = 0
    v = np.zeros((m,1))

    while it < i_max and error > min_error:
        if it_epoch >= N or it==0:
            it_epoch=0
            epoch += 1
            if epoch >= epochs:
                break
            shuffled_indexes = random.sample(range(0, N), N) # Shuffling the whole dataset before each epoch
            
        batch_start = it_epoch
        # Randomly selecting mb training examples
        random_indexes = tuple(shuffled_indexes[batch_start:batch_start+mb])
        xi = X[random_indexes,:]
        yi = y[random_indexes,:]

        
        yi_hat = xi @ a                      # funcao hipotese
        grad = (-2 * ((yi-yi_hat).T @ xi)).T # column vector of the gradient of the mse function
        # ADAGRAD
        v = v + grad**2
        a = a - (alpha_0/np.sqrt(v+epsilon)) * grad
        
        # Saving histories
        a_hist[it,:] = a.reshape(1,m)
        mse_hist[it] = (1/mb) * np.sum((yi-yi_hat)**2) # already saving the mse hist
        update_hist[it,:] = ((alpha_0/np.sqrt(v+epsilon)) * grad).reshape(1,m)
        v_hist[it,:] = v.reshape(1,m)

        #updating the error
        if it > 0:
            error = np.abs(mse_hist[it]-mse_hist[it-1])
        else:   
            error = mse_hist[0]
        
        it+=1
        it_epoch+=mb

    
    return a, a_hist, mse_hist, update_hist,v_hist


In [83]:
m = 1000
x1 = 3 * np.random.randn(m).reshape(m,1)
x2 = np.random.randn(m).reshape(m,1)
w = np.random.randn(m).reshape(m,1)
y_obj = 2*x1 + 1*x2
y_noisy = y_obj + w
X = np.c_[x1,x2]

a_opt = np.linalg.inv(X.T @ X) @ X.T @ y_noisy
print(a_opt)


[[1.99902691]
 [0.93386595]]


In [84]:
epochs = 10
mini_batch_size = 1 #1 = stochastic GD, m = classic GD
alpha_0 = 0.02
a, a_hist, mse_hist, update_hist,v_hist = minibatch_gradient_adagrad(X,y_noisy,epochs,m,mini_batch_size,alpha_0)
h = X @ a
print(f'pesos AdaGrad: \n{a}')
mse = (1/m) * np.sum((y_noisy-h)**2)
print(f'MSE AdaGrad: {mse}')

pesos AdaGrad: 
[[7.92641505]
 [8.58316854]]
MSE AdaGrad: 354.4061074138069


### RMSProp

https://chatgpt.com/c/68cef84d-d060-8329-9ff5-eb3feb1c9c7a

In [85]:
def minibatch_gradient_rmsprop(X,y,epochs,N,mb,alpha_0,epsilon=1e-8,beta=0.99999):
    """
    X: training examples
    y: training labels (y_noisy/funcao observavel)
    epochs: number of epochs
    N: number of examples
    mb: minibatch size
    alpha_0: initial learning rate/step size
    epsilon: escalar que evita divisão por 0
    beta: controla a memória da média móvel
    """
    i_max = epochs * (N // mb)
    min_error = 1e-3
    error = 1

    m = X.shape[1] #number of attributes
    a = 10 * np.ones(m).reshape(m,1) # initial column vector of weights

    a_hist = np.zeros((i_max,m)) # weight history
    mse_hist = np.zeros((i_max)) # error history
    update_hist = np.zeros((i_max,m))      # learning rate * gradient
    v_hist = np.zeros((i_max,m))    
    it = 0
    it_epoch = 0
    epoch = 0
    v = np.zeros((m,1))

    while it < i_max and error > min_error:
        if it_epoch >= N or it==0:
            it_epoch=0
            epoch += 1
            if epoch >= epochs:
                break
            shuffled_indexes = random.sample(range(0, N), N) # Shuffling the whole dataset before each epoch
            
        batch_start = it_epoch
        # Randomly selecting mb training examples
        random_indexes = tuple(shuffled_indexes[batch_start:batch_start+mb])
        xi = X[random_indexes,:]
        yi = y[random_indexes,:]

        
        yi_hat = xi @ a                      # funcao hipotese
        grad = (-2 * ((yi-yi_hat).T @ xi)).T # column vector of the gradient of the mse function
        # RMSPROP
        v = beta*v + (1-beta)*grad**2
        a = a - (alpha_0/np.sqrt(v+epsilon)) * grad
        
        # Saving histories
        a_hist[it,:] = a.reshape(1,m)
        mse_hist[it] = (1/mb) * np.sum((yi-yi_hat)**2) # already saving the mse hist
        update_hist[it,:] = ((alpha_0/np.sqrt(v+epsilon)) * grad).reshape(1,m)
        v_hist = v.reshape(1,m)   

        #updating the error
        if it > 0:
            error = np.abs(mse_hist[it]-mse_hist[it-1])
        else:   
            error = mse_hist[0]
        
        it+=1
        it_epoch+=mb

    
    return a, a_hist, mse_hist, update_hist


In [86]:
epochs = 10
mini_batch_size = 1 #1 = stochastic GD, m = classic GD
alpha_0 = 0.02
a, a_hist, mse_hist, update_hist = minibatch_gradient_rmsprop(X,y_noisy,epochs,m,mini_batch_size,alpha_0)
h = X @ a
print(f'pesos RMSProp: \n{a}')
mse = (1/m) * np.sum((y_noisy-h)**2)
print(f'MSE RMSProp: {mse}')

pesos RMSProp: 
[[1.87454578]
 [1.06638233]]
MSE RMSProp: 1.1384491343212109


### Adam

https://optimization.cbe.cornell.edu/index.php?title=Adam

In [87]:
def minibatch_gradient_adam(X,y,epochs,N,mb,alpha_0,epsilon=1e-8,beta1=0.9,beta2=0.999):
    """
    X: training examples
    y: training labels (y_noisy/funcao observavel)
    epochs: number of epochs
    N: number of examples
    mb: minibatch size
    alpha_0: initial learning rate/step size
    epsilon: escalar que evita divisão por 0
    beta: controla a memória da média móvel
    """
    i_max = epochs * (N // mb)
    min_error = 1e-3
    error = 1

    m = X.shape[1] #number of attributes
    a = 10 * np.ones(m).reshape(m,1) # initial column vector of weights

    a_hist = np.zeros((i_max,m)) # weight history
    mse_hist = np.zeros((i_max)) # error history
    update_hist = np.zeros((i_max,m))      # learning rate * gradient
    v_hist = np.zeros((i_max,m))    
    it = 0
    it_epoch = 0
    epoch = 0
    v = np.zeros((m,1))
    momentum = np.zeros((m,1))

    while it < i_max and error > min_error:
        if it_epoch >= N or it==0:
            it_epoch=0
            epoch += 1
            if epoch >= epochs:
                break
            shuffled_indexes = random.sample(range(0, N), N) # Shuffling the whole dataset before each epoch
            
        batch_start = it_epoch
        # Randomly selecting mb training examples
        random_indexes = tuple(shuffled_indexes[batch_start:batch_start+mb])
        xi = X[random_indexes,:]
        yi = y[random_indexes,:]

        
        yi_hat = xi @ a                      # funcao hipotese
        grad = (-2 * ((yi-yi_hat).T @ xi)).T # column vector of the gradient of the mse function
        # ADAM
        momentum = beta1 * momentum + (1-beta1) * grad
        v = beta2 * v + (1-beta2) * grad**2
        momentum_hat = momentum/(1-beta1)
        v_hat = v/(1-beta2)
        a = a - (alpha_0/np.sqrt(v_hat+epsilon)) * momentum_hat
        
        
        # Saving histories
        a_hist[it,:] = a.reshape(1,m)
        mse_hist[it] = (1/mb) * np.sum((yi-yi_hat)**2) # already saving the mse hist
        update_hist[it,:] = ((alpha_0/np.sqrt(v+epsilon)) * grad).reshape(1,m)
        v_hist = v.reshape(1,m)   

        #updating the error
        if it > 0:
            error = np.abs(mse_hist[it]-mse_hist[it-1])
        else:   
            error = mse_hist[0]
        
        it+=1
        it_epoch+=mb

    
    return a, a_hist, mse_hist, update_hist


In [88]:
epochs = 10
mini_batch_size = 1 #1 = stochastic GD, m = classic GD
alpha_0 = 0.02
a, a_hist, mse_hist, update_hist = minibatch_gradient_adam(X,y_noisy,epochs,m,mini_batch_size,alpha_0)
h = X @ a
print(f'pesos Adam: \n{a}')
mse = (1/m) * np.sum((y_noisy-h)**2)
print(f'MSE Adam: {mse}')

pesos Adam: 
[[1.99422759]
 [1.09538258]]
MSE Adam: 1.0303264507767522
